In [62]:
import torch
import random
import json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


SEED = 42

random.seed(SEED)
np.random.seed(SEED)


if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.13.0+cpu
CUDA available: False


In [63]:
import sys
import os
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nCurrent working directory:")
print(os.getcwd())

print("\nPath.cwd():")
print(Path.cwd())

print("\nD drive exists:")
print(Path("D:/").exists())

Python executable:
d:\razorpay_revenue_recovery\.venv\Scripts\python.exe

Current working directory:
d:\razorpay_revenue_recovery\ml\notebooks

Path.cwd():
d:\razorpay_revenue_recovery\ml\notebooks

D drive exists:
True


In [64]:
from pathlib import Path

DATASET_PATH = (
    Path.cwd().parent
    / "data"
    / "canonical_dataset.jsonl"
)

OUTPUT_DIR = (
    Path.cwd().parent
    / "models"
    / "query_classifier"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MAX_LENGTH = 128
TEST_SIZE = 0.15
VAL_SIZE = 0.15

print("Dataset:", DATASET_PATH)
print("Dataset exists:", DATASET_PATH.exists())
print("Output:", OUTPUT_DIR)

Dataset: d:\razorpay_revenue_recovery\ml\data\canonical_dataset.jsonl
Dataset exists: True
Output: d:\razorpay_revenue_recovery\ml\models\query_classifier


In [65]:
records = []

with DATASET_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    for line_number, line in enumerate(
        file,
        start=1,
    ):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Invalid JSON at line {line_number}"
            ) from exc

        records.append(record)

print(f"Loaded {len(records)} examples")

Loaded 1395 examples


In [66]:
import pandas as pd

df = pd.DataFrame(records)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (1395, 4)

Columns:
['query', 'intent', 'retrieval_mode', 'source_chunk_ids']


,query,intent,retrieval_mode,source_chunk_ids
0,Subscriptions: active state meaning,subscription_management,semantic,[8a1bf7083ee75a75_5e777323697c277a]
1,I received an API error; how should I interpre...,payment_failure_diagnostics,mixed,[2d717a7c18cc98ec_6747553425472b86]
2,How should I handle a Subscription that is `ex...,subscription_management,mixed,[325879c14bf5ef6b_1fbdff3023ecce1e]
3,I’m building an automated support flow for Fet...,payment_downtime,mixed,[8f8aacbf1a9b3d8c_9bed308dc04f5026]
4,I’m looking at Handle Failed Charge (Cards) in...,payment_retries,mixed,[db5897c980257b21_7cbdb852d6f14ddf]


In [67]:
print("INTENT DISTRIBUTION")
print("=" * 50)

print(
    df["intent"]
    .value_counts()
    .sort_index()
)

print("\nRETRIEVAL MODE DISTRIBUTION")
print("=" * 50)

print(
    df["retrieval_mode"]
    .value_counts()
    .sort_index()
)

INTENT DISTRIBUTION
intent
api_information                136
invoice_management             100
order_retrieval                105
payment_capture                113
payment_downtime               136
payment_failure_diagnostics    156
payment_retries                100
payment_retrieval              105
refund_management              136
subscription_management        172
webhook_management             136
Name: count, dtype: int64

RETRIEVAL MODE DISTRIBUTION
retrieval_mode
lexical     488
mixed       441
semantic    466
Name: count, dtype: int64


In [68]:
pd.crosstab(
    df["intent"],
    df["retrieval_mode"],
)

retrieval_mode,lexical,mixed,semantic
intent,,,
api_information,58,39,39
invoice_management,30,25,45
order_retrieval,37,33,35
payment_capture,34,38,41
payment_downtime,49,46,41
payment_failure_diagnostics,58,50,48
payment_retries,26,33,41
payment_retrieval,36,35,34
refund_management,54,38,44


In [69]:
from sklearn.model_selection import train_test_split

stratify_labels = (df["intent"] + "__" + df["retrieval_mode"])

train_val_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state= SEED, stratify=stratify_labels)

train_val_stratify = (train_val_df["intent"] + "__" + train_val_df["retrieval_mode"])

val_fraction_of_train_val = (VAL_SIZE/(1-TEST_SIZE))

train_df, val_df = train_test_split(train_val_df,
    test_size=val_fraction_of_train_val,
    random_state=SEED,
    stratify=train_val_stratify,)


print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Train: 975
Validation: 210
Test: 210
Total: 1395


In [70]:
print("TRAIN")
print(
    pd.crosstab(
        train_df["intent"],
        train_df["retrieval_mode"],
    )
)

print("\nVALIDATION")
print(
    pd.crosstab(
        val_df["intent"],
        val_df["retrieval_mode"],
    )
)

print("\nTEST")
print(
    pd.crosstab(
        test_df["intent"],
        test_df["retrieval_mode"],
    )
)

TRAIN
retrieval_mode               lexical  mixed  semantic
intent                                               
api_information                   40     27        27
invoice_management                21     17        31
order_retrieval                   25     23        25
payment_capture                   24     26        29
payment_downtime                  35     32        29
payment_failure_diagnostics       40     35        34
payment_retries                   18     23        29
payment_retrieval                 26     25        24
refund_management                 38     26        30
subscription_management           39     43        39
webhook_management                35     30        30

VALIDATION
retrieval_mode               lexical  mixed  semantic
intent                                               
api_information                    9      6         6
invoice_management                 4      4         7
order_retrieval                    6      5         5
payment_ca

In [71]:
train_queries = set(train_df["query"])
val_queries = set(val_df["query"])
test_queries = set(test_df["query"])

print(
    "Train ∩ Val:",
    len(train_queries & val_queries),
)

print(
    "Train ∩ Test:",
    len(train_queries & test_queries),
)

print(
    "Val ∩ Test:",
    len(val_queries & test_queries),
)

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [72]:
from sklearn.preprocessing import LabelEncoder

intent_encoder = LabelEncoder()

intent_encoder.fit(df["intent"])

intent_labels = list(intent_encoder.classes_)

print("Intent classes:")
for idx, label in enumerate(intent_labels):
    print(f"{idx:2d} -> {label}")

print(
    "\nNumber of intent classes:",
    len(intent_labels),
)



Intent classes:
 0 -> api_information
 1 -> invoice_management
 2 -> order_retrieval
 3 -> payment_capture
 4 -> payment_downtime
 5 -> payment_failure_diagnostics
 6 -> payment_retries
 7 -> payment_retrieval
 8 -> refund_management
 9 -> subscription_management
10 -> webhook_management

Number of intent classes: 11


In [73]:
retrieval_encoder = LabelEncoder()

retrieval_encoder.fit(df["retrieval_mode"])

retrieval_labels = list(retrieval_encoder.classes_)

print("Retrieval classes:")
for idx, label in enumerate(retrieval_labels):
    print(f"{idx:2d} -> {label}")

print("\n Number of intent classes:", len(retrieval_labels))

Retrieval classes:
 0 -> lexical
 1 -> mixed
 2 -> semantic

 Number of intent classes: 3


In [74]:
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

for split in (train_df, val_df, test_df):
    split["intent_id"] = intent_encoder.transform(split["intent"])

    split["retrieval_mode_id"] = retrieval_encoder.transform(split["retrieval_mode"])



In [75]:
train_df[
    [
        "query",
        "intent",
        "intent_id",
        "retrieval_mode",
        "retrieval_mode_id",
    ]
].head(10)

,query,intent,intent_id,retrieval_mode,retrieval_mode_id
1180,How would you route a support question specifi...,payment_downtime,4,semantic,2
593,I’m choosing a recurring payment method; what ...,subscription_management,9,mixed,1
383,How does /v1/invoices relate to Invoices?,invoice_management,1,mixed,1
678,What should a developer know about Payment Dow...,payment_downtime,4,lexical,0
752,Retry question — What happens on subscription ...,subscription_management,9,semantic,2
158,How do I troubleshoot a failure using the Comm...,payment_failure_diagnostics,5,mixed,1
516,I need to troubleshoot php: PHP\n$api = new Ap...,payment_retrieval,7,mixed,1
299,Can you explain Payment Retries for a payment ...,payment_retries,6,mixed,1
818,What happens if I try to capture an already ca...,payment_capture,3,semantic,2
393,How would I troubleshoot an issue involving Fe...,order_retrieval,2,semantic,2


In [76]:
from transformers import(AutoTokenizer, AutoModel,)

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

encoder = AutoModel.from_pretrained(MODEL_NAME)
print("Model:", MODEL_NAME)
print("Hidden size:", encoder.config.hidden_size,)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1656.13it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: distilbert-base-uncased
Hidden size: 768


In [77]:
MAX_LENGTH = 128

In [78]:
token_lengths = []

for query in train_df["query"]:
    tokens = tokenizer(query, truncation=False, add_special_tokens=True)["input_ids"]
    token_lengths.append(len(tokens))

lengths = np.array(token_lengths)

print("Maximum tokens:", lengths.max())
print("Mean tokens:", lengths.mean())
print("Median tokens:", np.median(lengths))
print("95th percentile:", np.percentile(lengths, 95))
print("99th percentile:", np.percentile(lengths, 99))

Maximum tokens: 278
Mean tokens: 20.74871794871795
Median tokens: 16.0
95th percentile: 39.0
99th percentile: 98.77999999999997


In [79]:
print(
    "Queries > 128 tokens:",
    (lengths > MAX_LENGTH).sum(),
)

print(
    "Percentage > 128:",
    100 * (lengths > MAX_LENGTH).mean(),
)

Queries > 128 tokens: 6
Percentage > 128: 0.6153846153846154


In [80]:
sample_query = train_df.iloc[0]["query"]

encoded = tokenizer(sample_query,
    truncation=True,
    max_length=MAX_LENGTH,
    padding="max_length",)

print("Query:")
print(sample_query)

print("\nInput IDs:")
print(encoded["input_ids"][:20])

print("\nAttention mask:")
print(encoded["attention_mask"][:20])

Query:
How would you route a support question specifically about Response? I’m trying to use the `json: UPI PSP
{
  "id": "down_F8LCfthx90fMOo",
  "method": "upi",
  "begin": 1593412063,
  "end": null,
  "status": "started",
  "scheduled": false,
  "severity": "high",
  "instrument": {
    "psp": "bhim",
    "flow": "collect"
  },
  "created_at": 1593412092,
  "updated_at": 1593412092
}` detail correctly.

Input IDs:
[101, 2129, 2052, 2017, 2799, 1037, 2490, 3160, 4919, 2055, 3433, 1029, 1045, 1521, 1049, 2667, 2000, 2224, 1996, 1036]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [81]:
print(
    "\nDecoded:"
)

print(
    tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=False,
    )
)


Decoded:
[CLS] how would you route a support question specifically about response? i ’ m trying to use the ` json : upi psp { " id " : " down _ f8lcfthx90fmoo ", " method " : " upi ", " begin " : 1593412063, " end " : null, " status " : " started ", " scheduled " : false, " severity " : " high ", " instrument " : { " psp " : " bhim ", " flow " : " collect " }, " created _ at " : 1593412 [SEP]


In [82]:
from torch.utils.data import Dataset


class QueryClassificationDataset(Dataset):

    def __init__(
        self,
        dataframe: pd.DataFrame,
        tokenizer,
        max_length: int,
    ) -> None:

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(
        self,
        index: int,
    ) -> dict[str, torch.Tensor]:

        row = self.dataframe.iloc[index]

        encoding = self.tokenizer(
            row["query"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding[
                "attention_mask"
            ].squeeze(0),
            "intent_id": torch.tensor(
                row["intent_id"],
                dtype=torch.long,
            ),
            "retrieval_mode_id": torch.tensor(
                row["retrieval_mode_id"],
                dtype=torch.long,
            ),
        }

In [83]:
train_dataset = QueryClassificationDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

val_dataset = QueryClassificationDataset(
    dataframe=val_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

test_dataset = QueryClassificationDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 975
Validation: 210
Test: 210


In [84]:
sample = train_dataset[0]

for key, value in sample.items():
    print(
        f"{key:20s}",
        value.shape,
        value.dtype,
    )

input_ids            torch.Size([128]) torch.int64
attention_mask       torch.Size([128]) torch.int64
intent_id            torch.Size([]) torch.int64
retrieval_mode_id    torch.Size([]) torch.int64


In [85]:
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=True,
    )
)

print(
    "Intent ID:",
    sample["intent_id"].item(),
)

print(
    "Retrieval ID:",
    sample["retrieval_mode_id"].item(),
)

how would you route a support question specifically about response? i ’ m trying to use the ` json : upi psp { " id " : " down _ f8lcfthx90fmoo ", " method " : " upi ", " begin " : 1593412063, " end " : null, " status " : " started ", " scheduled " : false, " severity " : " high ", " instrument " : { " psp " : " bhim ", " flow " : " collect " }, " created _ at " : 1593412
Intent ID: 4
Retrieval ID: 2


In [86]:
train_lengths = []

for idx, query in enumerate(train_df["query"]):
    token_count = len(
        tokenizer(
            query,
            truncation=False,
            add_special_tokens=True,
        )["input_ids"]
    )

    train_lengths.append(
        (idx, token_count, query)
    )

longest_queries = sorted(
    train_lengths,
    key=lambda x: x[1],
    reverse=True,
)[:10]

for idx, token_count, query in longest_queries:
    print("=" * 80)
    print(f"Index: {idx}")
    print(f"Tokens: {token_count}")
    print(query)

Index: 904
Tokens: 278
I’m investigating a transaction and found Fetch All Orders; what does it refer to? I’m specifically looking at `json: Success
{
  "entity": "collection",
  "count": 2,
  "items": [
    {
      "id": "order_EKzX2WiEWbMxmx",
      "entity": "order",
      "amount": 1234,
      "amount_paid": 0,
      "amount_due": 1234,
      "currency": "INR",
      "receipt": "Receipt No. 1",
      "offer_id": null,
      "status": "created",
      "attempts": 0,
      "notes": [],
      "created_at": 1582637108
    },
    {
      "id": "order_EAI5nRfThga2TU",
      "entity": "order",
      "amount": 100,
      "amount_paid": 0,
      "amount_due": 100,
      "currency": "INR",
      "receipt": "Receipt No. 1",
      "offer_id": null,
      "status": "created",
      "attempts": 0,
      "notes": [],
      "created_at": 1580300731
    }
  ]
}

`.
Index: 942
Tokens: 164
How does json: UPI Handle
{
    "entity": "collection",
    "count": 1,
    "items": [
      {
        "id": "do

In [87]:
# Analyze token lengths across the training set

length_records = []

for idx, row in train_df.iterrows():
    token_ids = tokenizer(
        row["query"],
        truncation=False,
        add_special_tokens=True,
    )["input_ids"]

    length_records.append({
        "index": idx,
        "tokens": len(token_ids),
        "intent": row["intent"],
        "retrieval_mode": row["retrieval_mode"],
        "query": row["query"],
    })

length_df = pd.DataFrame(length_records)

print("Token length statistics")
print("=" * 60)

print(length_df["tokens"].describe())

print("\nQueries over 64 tokens:")
print((length_df["tokens"] > 64).sum())

print("\nQueries over 96 tokens:")
print((length_df["tokens"] > 96).sum())

print("\nQueries over 128 tokens:")
print((length_df["tokens"] > 128).sum())

Token length statistics
count    975.000000
mean      20.748718
std       17.733582
min        4.000000
25%       13.000000
50%       16.000000
75%       24.000000
max      278.000000
Name: tokens, dtype: float64

Queries over 64 tokens:
25

Queries over 96 tokens:
11

Queries over 128 tokens:
6


In [88]:
long_queries = (
    length_df[
        length_df["tokens"] > 96
    ]
    .sort_values(
        "tokens",
        ascending=False,
    )
)

for _, row in long_queries.iterrows():
    print("=" * 80)
    print(
        f"Tokens: {row['tokens']} | "
        f"Intent: {row['intent']} | "
        f"Mode: {row['retrieval_mode']}"
    )
    print(row["query"])

Tokens: 278 | Intent: order_retrieval | Mode: semantic
I’m investigating a transaction and found Fetch All Orders; what does it refer to? I’m specifically looking at `json: Success
{
  "entity": "collection",
  "count": 2,
  "items": [
    {
      "id": "order_EKzX2WiEWbMxmx",
      "entity": "order",
      "amount": 1234,
      "amount_paid": 0,
      "amount_due": 1234,
      "currency": "INR",
      "receipt": "Receipt No. 1",
      "offer_id": null,
      "status": "created",
      "attempts": 0,
      "notes": [],
      "created_at": 1582637108
    },
    {
      "id": "order_EAI5nRfThga2TU",
      "entity": "order",
      "amount": 100,
      "amount_paid": 0,
      "amount_due": 100,
      "currency": "INR",
      "receipt": "Receipt No. 1",
      "offer_id": null,
      "status": "created",
      "attempts": 0,
      "notes": [],
      "created_at": 1580300731
    }
  ]
}

`.
Tokens: 164 | Intent: payment_downtime | Mode: mixed
How does json: UPI Handle
{
    "entity": "collect

In [89]:
from typing import Any

import torch
import torch.nn as nn

from transformers import AutoModel

class MultiTaskQueryClassifier(nn.Module):

    def __init__(self, model_name: str, num_intents: int, num_reterival_modes: int, dropout: float =0.2) -> None:

        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = (self.encoder.config.hidden_size)

        self.dropout = nn.Dropout(dropout)

        self.intent_classifier = nn.Linear(hidden_size, num_intents)

        self.retrieval_classifier = nn.Linear(hidden_size, num_reterival_modes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:

        outputs = self.encoder(input_ids = input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:,0]

        pooled = self.dropout(pooled)

        intent_logits = (self.intent_classifier(pooled))

        retrieval_logits = (self.retrieval_classifier(pooled))

        return (intent_logits, retrieval_logits)

In [90]:
NUM_INTENTS = len(intent_encoder.classes_)
NUM_RETRIEVAL_MODES = len(retrieval_encoder.classes_)

device = torch.device("cuda"
                      if torch.cuda.is_available()
                      else "cpu"
                      )

model = MultiTaskQueryClassifier(model_name=MODEL_NAME, num_intents=NUM_INTENTS, num_reterival_modes = NUM_RETRIEVAL_MODES)

model = model.to(device)

print("Device:", device)
print("Intent classes:", NUM_INTENTS)
print("Retrieval classes", NUM_RETRIEVAL_MODES)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2953.32it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cpu
Intent classes: 11
Retrieval classes 3


In [91]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:", f"{total_params}")
print(f"Trainable parameters:", f"{trainable_params}")

Total parameters: 66373646
Trainable parameters: 66373646


In [92]:
train_df.head()

,query,intent,retrieval_mode,source_chunk_ids,intent_id,retrieval_mode_id
1180,How would you route a support question specifi...,payment_downtime,semantic,[943825c4a71626b7_9553e43272698f6d],4,2
593,I’m choosing a recurring payment method; what ...,subscription_management,mixed,[7fd15ccda574d5c6_b4e2728bb89b59c8],9,1
383,How does /v1/invoices relate to Invoices?,invoice_management,mixed,[30267c389e8c3073_77c2bc58c00d673b],1,1
678,What should a developer know about Payment Dow...,payment_downtime,lexical,[f6ec2f648af712bc_c0da15c1a1af0fcd],4,0
752,Retry question — What happens on subscription ...,subscription_management,semantic,"[561203620086ce01_0889c7f99397bc15, c161f2762c...",9,2


In [94]:
sample = train_dataset[0]

input_ids = (sample["input_ids"].unsqueeze(0).to(device))

attention_mask = (sample["attention_mask"].unsqueeze(0).to(device))

with torch.no_grad():
    intent_logits, retrieval_logits = (model(input_ids=input_ids, attention_mask=attention_mask))

print("Intent logits:",
    intent_logits.shape,)


print(
    "Retrieval logits:",
    retrieval_logits.shape,
)

Intent logits: torch.Size([1, 11])
Retrieval logits: torch.Size([1, 3])


In [96]:
train_dataset[0]

{'input_ids': tensor([  101,  2129,  2052,  2017,  2799,  1037,  2490,  3160,  4919,  2055,
          3433,  1029,  1045,  1521,  1049,  2667,  2000,  2224,  1996,  1036,
          1046,  3385,  1024,  2039,  2072,  8827,  2361,  1063,  1000,  8909,
          1000,  1024,  1000,  2091,  1035,  1042,  2620, 15472,  6199,  2232,
          2595, 21057, 16715,  9541,  1000,  1010,  1000,  4118,  1000,  1024,
          1000,  2039,  2072,  1000,  1010,  1000,  4088,  1000,  1024, 18914,
         22022, 12521,  2692,  2575,  2509,  1010,  1000,  2203,  1000,  1024,
         19701,  1010,  1000,  3570,  1000,  1024,  1000,  2318,  1000,  1010,
          1000,  5115,  1000,  1024,  6270,  1010,  1000, 18976,  1000,  1024,
          1000,  2152,  1000,  1010,  1000,  6602,  1000,  1024,  1063,  1000,
          8827,  2361,  1000,  1024,  1000,  1038, 14341,  1000,  1010,  1000,
          4834,  1000,  1024,  1000,  8145,  1000,  1065,  1010,  1000,  2580,
          1035,  2012,  1000,  1024, 18

In [98]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True,)

val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle = True,)

test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = True,)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


Train batches: 61
Validation batches: 14
Test batches: 14


In [100]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(f"{key:22s}",
        value.shape,
        value.dtype,)

input_ids              torch.Size([16, 128]) torch.int64
attention_mask         torch.Size([16, 128]) torch.int64
intent_id              torch.Size([16]) torch.int64
retrieval_mode_id      torch.Size([16]) torch.int64


In [101]:
intent_loss_fn = nn.CrossEntropyLoss()

retrieval_loss_fn = nn.CrossEntropyLoss()

In [102]:
from torch.optim import AdamW

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

OPTIMIZER = AdamW(model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,)

print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Learning rate: 2e-05
Weight decay: 0.01


In [103]:
EPOCHS = 4

WARMUP_RATIO = 0.1

TOTAL_TRAINING_STEPS = (
    len(train_loader) * EPOCHS
)

WARMUP_STEPS = int(
    TOTAL_TRAINING_STEPS * WARMUP_RATIO
)

print("Epochs:", EPOCHS)
print("Training steps:", TOTAL_TRAINING_STEPS)
print("Warmup steps:", WARMUP_STEPS)

Epochs: 4
Training steps: 244
Warmup steps: 24


In [105]:
from transformers import get_linear_schedule_with_warmup

scheduler = get_linear_schedule_with_warmup(
    OPTIMIZER,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_TRAINING_STEPS,
)

In [113]:
from tqdm.auto import tqdm


def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    intent_loss_fn,
    retrieval_loss_fn,
    device,
):
    model.train()

    total_loss = 0.0
    total_intent_loss = 0.0
    total_retrieval_loss = 0.0

    progress_bar = tqdm(
        dataloader,
        desc="Training",
        leave=False,
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        intent_targets = batch[
            "intent_id"
        ].to(device)

        retrieval_targets = batch[
            "retrieval_mode_id"
        ].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        intent_logits, retrieval_logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # Calculate the two task losses
        intent_loss = intent_loss_fn(
            intent_logits,
            intent_targets,
        )

        retrieval_loss = retrieval_loss_fn(
            retrieval_logits,
            retrieval_targets,
        )

        # Combined multi-task loss
        loss = (
            0.5*intent_loss
            + 1.5*retrieval_loss
        )

        # Backpropagation
        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        # Update model parameters
        optimizer.step()

        # Update learning rate
        scheduler.step()

        # Accumulate losses
        total_loss += loss.item()
        total_intent_loss += intent_loss.item()
        total_retrieval_loss += retrieval_loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    num_batches = len(dataloader)

    return {
        "loss": total_loss / num_batches,
        "intent_loss": (
            total_intent_loss / num_batches
        ),
        "retrieval_loss": (
            total_retrieval_loss / num_batches
        ),
    }

In [108]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
)


def evaluate(
    model,
    dataloader,
    intent_loss_fn,
    retrieval_loss_fn,
    device,
):
    model.eval()

    total_loss = 0.0
    total_intent_loss = 0.0
    total_retrieval_loss = 0.0

    all_intent_targets = []
    all_intent_predictions = []

    all_retrieval_targets = []
    all_retrieval_predictions = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch[
                "input_ids"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            intent_targets = batch[
                "intent_id"
            ].to(device)

            retrieval_targets = batch[
                "retrieval_mode_id"
            ].to(device)

            # Forward pass
            intent_logits, retrieval_logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            # Losses
            intent_loss = intent_loss_fn(
                intent_logits,
                intent_targets,
            )

            retrieval_loss = retrieval_loss_fn(
                retrieval_logits,
                retrieval_targets,
            )

            loss = (
                intent_loss
                + retrieval_loss
            )

            total_loss += loss.item()
            total_intent_loss += (
                intent_loss.item()
            )
            total_retrieval_loss += (
                retrieval_loss.item()
            )

            # Predictions
            intent_predictions = (
                torch.argmax(
                    intent_logits,
                    dim=1,
                )
            )

            retrieval_predictions = (
                torch.argmax(
                    retrieval_logits,
                    dim=1,
                )
            )

            # Move predictions to CPU
            all_intent_targets.extend(
                intent_targets.cpu().numpy()
            )

            all_intent_predictions.extend(
                intent_predictions.cpu().numpy()
            )

            all_retrieval_targets.extend(
                retrieval_targets.cpu().numpy()
            )

            all_retrieval_predictions.extend(
                retrieval_predictions.cpu().numpy()
            )

    num_batches = len(dataloader)

    return {
        "loss": total_loss / num_batches,

        "intent_loss": (
            total_intent_loss
            / num_batches
        ),

        "retrieval_loss": (
            total_retrieval_loss
            / num_batches
        ),

        "intent_accuracy": accuracy_score(
            all_intent_targets,
            all_intent_predictions,
        ),

        "intent_macro_f1": f1_score(
            all_intent_targets,
            all_intent_predictions,
            average="macro",
        ),

        "retrieval_accuracy": accuracy_score(
            all_retrieval_targets,
            all_retrieval_predictions,
        ),

        "retrieval_macro_f1": f1_score(
            all_retrieval_targets,
            all_retrieval_predictions,
            average="macro",
        ),
    }

In [109]:
initial_metrics = evaluate(
    model=model,
    dataloader=val_loader,
    intent_loss_fn=intent_loss_fn,
    retrieval_loss_fn=retrieval_loss_fn,
    device=device,
)

print(
    f"Loss: {initial_metrics['loss']:.4f}"
)

print(
    f"Intent accuracy: "
    f"{initial_metrics['intent_accuracy']:.4f}"
)

print(
    f"Intent macro-F1: "
    f"{initial_metrics['intent_macro_f1']:.4f}"
)

print(
    f"Retrieval accuracy: "
    f"{initial_metrics['retrieval_accuracy']:.4f}"
)

print(
    f"Retrieval macro-F1: "
    f"{initial_metrics['retrieval_macro_f1']:.4f}"
)

Loss: 3.5216
Intent accuracy: 0.0429
Intent macro-F1: 0.0158
Retrieval accuracy: 0.3476
Retrieval macro-F1: 0.1720


In [ ]:
import copy
import time


BEST_MODEL_PATH = OUTPUT_DIR / "best_model_reterival_tuned.pt"

PATIENCE = 2

best_val_f1 = -float("inf")
epochs_without_improvement = 0

history = []

for epoch in range(1, EPOCHS + 1):

    print()
    print("=" * 70)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("=" * 70)

    start_time = time.time()

    # -------------------------
    # Training
    # -------------------------

    train_metrics = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=OPTIMIZER,
        scheduler=scheduler,
        intent_loss_fn=intent_loss_fn,
        retrieval_loss_fn=retrieval_loss_fn,
        device=device,
    )

    # -------------------------
    # Validation
    # -------------------------

    val_metrics = evaluate(
        model=model,
        dataloader=val_loader,
        intent_loss_fn=intent_loss_fn,
        retrieval_loss_fn=retrieval_loss_fn,
        device=device,
    )

    epoch_time = time.time() - start_time

    # Combined metric
    #
    # Intent is the primary routing decision,
    # while retrieval mode is the secondary decision.
    combined_f1 = (
        0.7 * val_metrics["intent_macro_f1"]
        + 0.3 * val_metrics["retrieval_macro_f1"]
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_intent_loss": train_metrics["intent_loss"],
        "train_retrieval_loss": train_metrics[
            "retrieval_loss"
        ],
        "val_loss": val_metrics["loss"],
        "val_intent_accuracy": val_metrics[
            "intent_accuracy"
        ],
        "val_intent_macro_f1": val_metrics[
            "intent_macro_f1"
        ],
        "val_retrieval_accuracy": val_metrics[
            "retrieval_accuracy"
        ],
        "val_retrieval_macro_f1": val_metrics[
            "retrieval_macro_f1"
        ],
        "combined_f1": combined_f1,
        "epoch_time": epoch_time,
    })

    print(
        f"Train loss:              "
        f"{train_metrics['loss']:.4f}"
    )

    print(
        f"Validation loss:         "
        f"{val_metrics['loss']:.4f}"
    )

    print(
        f"Intent accuracy:         "
        f"{val_metrics['intent_accuracy']:.4f}"
    )

    print(
        f"Intent macro-F1:         "
        f"{val_metrics['intent_macro_f1']:.4f}"
    )

    print(
        f"Retrieval accuracy:      "
        f"{val_metrics['retrieval_accuracy']:.4f}"
    )

    print(
        f"Retrieval macro-F1:      "
        f"{val_metrics['retrieval_macro_f1']:.4f}"
    )

    print(
        f"Combined validation F1:  "
        f"{combined_f1:.4f}"
    )

    print(
        f"Epoch time:              "
        f"{epoch_time:.1f}s"
    )

    # -------------------------
    # Checkpoint
    # -------------------------

    if combined_f1 > best_val_f1:

        best_val_f1 = combined_f1
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": MODEL_NAME,
                "num_intents": NUM_INTENTS,
                "num_retrieval_modes": (
                    NUM_RETRIEVAL_MODES
                ),
                "max_length": MAX_LENGTH,
                "best_val_f1": best_val_f1,
            },
            BEST_MODEL_PATH,
        )

        print(
            f"✓ New best model saved "
            f"(F1={best_val_f1:.4f})"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement "
            f"({epochs_without_improvement}/"
            f"{PATIENCE})"
        )

        if epochs_without_improvement >= PATIENCE:

            print(
                "\nEarly stopping triggered."
            )

            break


Epoch 1/4


Train loss:              3.3953
Validation loss:         2.9491
Intent accuracy:         0.4667
Intent macro-F1:         0.3577
Retrieval accuracy:      0.5286
Retrieval macro-F1:      0.5110
Combined validation F1:  0.4037
Epoch time:              299.7s
✓ New best model saved (F1=0.4037)

Epoch 2/4


Train loss:              2.3536
Validation loss:         1.8282
Intent accuracy:         0.8190
Intent macro-F1:         0.8122
Retrieval accuracy:      0.6286
Retrieval macro-F1:      0.6258
Combined validation F1:  0.7563
Epoch time:              304.1s
✓ New best model saved (F1=0.7563)

Epoch 3/4


Train loss:              1.5716
Validation loss:         1.4166
Intent accuracy:         0.8571
Intent macro-F1:         0.8533
Retrieval accuracy:      0.6714
Retrieval macro-F1:      0.6695
Combined validation F1:  0.7981
Epoch time:              308.1s
✓ New best model saved (F1=0.7981)

Epoch 4/4


Train loss:              1.2757
Validation loss:         1.3289
Intent accuracy:         0.8714
Intent macro-F1:         0.8695
Retrieval accuracy:      0.6810
Retrieval macro-F1:      0.6797
Combined validation F1:  0.8126
Epoch time:              305.2s
✓ New best model saved (F1=0.8126)


In [112]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

retrieval_classes = np.arange(NUM_RETRIEVAL_MODES)

retrieval_weights = compute_class_weight(class_weight="balanced", classes=retrieval_classes, y = train_df["retrieval_mode_id"])

retrieval_weights = torch.tensor(retrieval_weights, dtype=torch.float32,).to(device)

print("Retrieval class weights:")

for class_id, weight in zip(
    retrieval_classes,
    retrieval_weights.cpu().numpy(),
):
    print(
        f"{retrieval_labels[class_id]:10s} → {weight:.4f}"
    )

Retrieval class weights:
lexical    → 0.9531
mixed      → 1.0586
semantic   → 0.9939


In [ ]:
model = MultiTaskQueryClassifier(
    model_name=MODEL_NAME,
    num_intents=NUM_INTENTS,
    num_reterival_modes=NUM_RETRIEVAL_MODES,
)

model = model.to(device)

print("Experiment B: fresh model initialized.")

In [114]:
optimizer = AdamW(model.parameters(), lr = LEARNING_RATE, weight_decay=WEIGHT_DECAY)

In [115]:
TOTAL_TRAINING_STEPS = len(train_loader) * EPOCHS

WARMUP_STEPS = int(
    TOTAL_TRAINING_STEPS * WARMUP_RATIO
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_TRAINING_STEPS,
)

In [116]:
BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "best_model_retrieval_tuned.pt"
)

print(
    "Experiment B checkpoint:",
    BEST_MODEL_PATH,
)

Experiment B checkpoint: d:\razorpay_revenue_recovery\ml\models\query_classifier\best_model_retrieval_tuned.pt


In [117]:
import copy
import time



PATIENCE = 2

best_val_f1 = -float("inf")
epochs_without_improvement = 0

history = []

for epoch in range(1, EPOCHS + 1):

    print()
    print("=" * 70)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("=" * 70)

    start_time = time.time()

    # -------------------------
    # Training
    # -------------------------

    train_metrics = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=OPTIMIZER,
        scheduler=scheduler,
        intent_loss_fn=intent_loss_fn,
        retrieval_loss_fn=retrieval_loss_fn,
        device=device,
    )

    # -------------------------
    # Validation
    # -------------------------

    val_metrics = evaluate(
        model=model,
        dataloader=val_loader,
        intent_loss_fn=intent_loss_fn,
        retrieval_loss_fn=retrieval_loss_fn,
        device=device,
    )

    epoch_time = time.time() - start_time

    # Combined metric
    #
    # Intent is the primary routing decision,
    # while retrieval mode is the secondary decision.
    combined_f1 = (
        0.5 * val_metrics["intent_macro_f1"]
        + 0.5 * val_metrics["retrieval_macro_f1"]
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_intent_loss": train_metrics["intent_loss"],
        "train_retrieval_loss": train_metrics[
            "retrieval_loss"
        ],
        "val_loss": val_metrics["loss"],
        "val_intent_accuracy": val_metrics[
            "intent_accuracy"
        ],
        "val_intent_macro_f1": val_metrics[
            "intent_macro_f1"
        ],
        "val_retrieval_accuracy": val_metrics[
            "retrieval_accuracy"
        ],
        "val_retrieval_macro_f1": val_metrics[
            "retrieval_macro_f1"
        ],
        "combined_f1": combined_f1,
        "epoch_time": epoch_time,
    })

    print(
        f"Train loss:              "
        f"{train_metrics['loss']:.4f}"
    )

    print(
        f"Validation loss:         "
        f"{val_metrics['loss']:.4f}"
    )

    print(
        f"Intent accuracy:         "
        f"{val_metrics['intent_accuracy']:.4f}"
    )

    print(
        f"Intent macro-F1:         "
        f"{val_metrics['intent_macro_f1']:.4f}"
    )

    print(
        f"Retrieval accuracy:      "
        f"{val_metrics['retrieval_accuracy']:.4f}"
    )

    print(
        f"Retrieval macro-F1:      "
        f"{val_metrics['retrieval_macro_f1']:.4f}"
    )

    print(
        f"Combined validation F1:  "
        f"{combined_f1:.4f}"
    )

    print(
        f"Epoch time:              "
        f"{epoch_time:.1f}s"
    )

    # -------------------------
    # Checkpoint
    # -------------------------

    if combined_f1 > best_val_f1:

        best_val_f1 = combined_f1
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": MODEL_NAME,
                "num_intents": NUM_INTENTS,
                "num_retrieval_modes": (
                    NUM_RETRIEVAL_MODES
                ),
                "max_length": MAX_LENGTH,
                "best_val_f1": best_val_f1,
            },
            BEST_MODEL_PATH,
        )

        print(
            f"✓ New best model saved "
            f"(F1={best_val_f1:.4f})"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement "
            f"({epochs_without_improvement}/"
            f"{PATIENCE})"
        )

        if epochs_without_improvement >= PATIENCE:

            print(
                "\nEarly stopping triggered."
            )

            break


Epoch 1/4


Training:   0%|          | 0/61 [00:00<?, ?it/s]C:\Users\sumed\AppData\Local\Temp\ipykernel_17348\3893850.py:77: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Train loss:              1.2262
Validation loss:         1.3675
Intent accuracy:         0.8714
Intent macro-F1:         0.8695
Retrieval accuracy:      0.6810
Retrieval macro-F1:      0.6797
Combined validation F1:  0.7746
Epoch time:              227.4s
✓ New best model saved (F1=0.7746)

Epoch 2/4


Train loss:              1.2294
Validation loss:         1.3270
Intent accuracy:         0.8714
Intent macro-F1:         0.8695
Retrieval accuracy:      0.6810
Retrieval macro-F1:      0.6797
Combined validation F1:  0.7746
Epoch time:              464.8s
No improvement (1/2)

Epoch 3/4


Train loss:              1.2135
Validation loss:         1.3633
Intent accuracy:         0.8714
Intent macro-F1:         0.8695
Retrieval accuracy:      0.6810
Retrieval macro-F1:      0.6797
Combined validation F1:  0.7746
Epoch time:              522.8s
No improvement (2/2)

Early stopping triggered.


In [121]:
BEST_MODEL_PATH = (OUTPUT_DIR / "best_model.pt")

check_point = torch.load(BEST_MODEL_PATH, map_location=device)

model.load_state_dict(check_point["model_state_dict"])

model = model.to(device)

print("Loaded:", BEST_MODEL_PATH)
print(
    "Best validation F1:",
    check_point["best_val_f1"],
)

Loaded: d:\razorpay_revenue_recovery\ml\models\query_classifier\best_model.pt
Best validation F1: 0.8125727120436821


In [122]:
test_metrics = evaluate(
    model=model,
    dataloader=test_loader,
    intent_loss_fn=intent_loss_fn,
    retrieval_loss_fn=retrieval_loss_fn,
    device=device,
)

print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(
    f"Test loss:             "
    f"{test_metrics['loss']:.4f}"
)

print(
    f"Intent accuracy:       "
    f"{test_metrics['intent_accuracy']:.4f}"
)

print(
    f"Intent macro-F1:       "
    f"{test_metrics['intent_macro_f1']:.4f}"
)

print(
    f"Retrieval accuracy:    "
    f"{test_metrics['retrieval_accuracy']:.4f}"
)

print(
    f"Retrieval macro-F1:    "
    f"{test_metrics['retrieval_macro_f1']:.4f}"
)

FINAL TEST RESULTS
Test loss:             1.5281
Intent accuracy:       0.8476
Intent macro-F1:       0.8468
Retrieval accuracy:    0.7048
Retrieval macro-F1:    0.7057


In [123]:
from sklearn.metrics import classification_report

# We need predictions from the test set
model.eval()

all_intent_targets = []
all_intent_predictions = []

all_retrieval_targets = []
all_retrieval_predictions = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        intent_targets = batch[
            "intent_id"
        ].to(device)

        retrieval_targets = batch[
            "retrieval_mode_id"
        ].to(device)

        intent_logits, retrieval_logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        intent_predictions = torch.argmax(
            intent_logits,
            dim=1,
        )

        retrieval_predictions = torch.argmax(
            retrieval_logits,
            dim=1,
        )

        all_intent_targets.extend(
            intent_targets.cpu().numpy()
        )

        all_intent_predictions.extend(
            intent_predictions.cpu().numpy()
        )

        all_retrieval_targets.extend(
            retrieval_targets.cpu().numpy()
        )

        all_retrieval_predictions.extend(
            retrieval_predictions.cpu().numpy()
        )

In [124]:
print("=" * 70)
print("INTENT CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_intent_targets,
        all_intent_predictions,
        target_names=intent_labels,
        digits=4,
    )
)

print("=" * 70)
print("RETRIEVAL MODE CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_retrieval_targets,
        all_retrieval_predictions,
        target_names=retrieval_labels,
        digits=4,
    )
)

INTENT CLASSIFICATION REPORT
                             precision    recall  f1-score   support

            api_information     0.7692    0.4762    0.5882        21
         invoice_management     0.8889    1.0000    0.9412        16
            order_retrieval     0.8125    0.8125    0.8125        16
            payment_capture     0.8889    0.9412    0.9143        17
           payment_downtime     1.0000    0.6500    0.7879        20
payment_failure_diagnostics     0.7667    0.9583    0.8519        24
            payment_retries     0.8667    0.8667    0.8667        15
          payment_retrieval     0.8333    1.0000    0.9091        15
          refund_management     0.9500    0.9048    0.9268        21
    subscription_management     0.7419    0.9200    0.8214        25
         webhook_management     0.9444    0.8500    0.8947        20

                   accuracy                         0.8476       210
                  macro avg     0.8602    0.8527    0.8468       210
   

In [125]:
from typing import Any

import torch 
import torch.nn as nn

class ONNXQueryClassifier(nn.Module):

    def __init__(self, model) -> None:
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        intent_logits, retrieval_logits = self.model(input_ids = input_ids, attention_mask = attention_mask)

        return (intent_logits, retrieval_logits)

In [126]:
BEST_MODEL_PATH = (
    OUTPUT_DIR / "best_model.pt"
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to("cpu")
model.eval()

print("Loaded production checkpoint:")
print(BEST_MODEL_PATH)

Loaded production checkpoint:
d:\razorpay_revenue_recovery\ml\models\query_classifier\best_model.pt


In [127]:
ONNX_DIR = (OUTPUT_DIR/ "onnx")

ONNX_DIR.mkdir(parents=True, exist_ok=True)

ONNX_PATH = (
    ONNX_DIR / "model.onnx"
)

print("ONNX output:")
print(ONNX_PATH)

ONNX output:
d:\razorpay_revenue_recovery\ml\models\query_classifier\onnx\model.onnx


In [129]:
onnx_model = ONNXQueryClassifier(model)

dummy_input_ids = torch.ones((1, MAX_LENGTH), dtype=torch.long)

dummy_attention_mask = torch.ones((1, MAX_LENGTH), dtype=torch.long)

torch.onnx.export(
    onnx_model,(dummy_input_ids, dummy_attention_mask), ONNX_PATH, input_names=["input_ids","attention_mask"], output_names=["intent_logits","retrieval_logits",],
    dynamic_axes={
        "input_ids": {
            0: "batch_size",
            1: "sequence_length",
        },
        "attention_mask": {
            0: "batch_size",
            1: "sequence_length",
        },
        "intent_logits": {
            0: "batch_size",
        },
        "retrieval_logits": {
            0: "batch_size",
        },
    },
    opset_version=17,
)

print(
    f"ONNX model exported to:\n{ONNX_PATH}"
)

C:\Users\sumed\AppData\Local\Temp\ipykernel_17348\159985506.py:7: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
C:\Users\sumed\AppData\Local\Temp\ipykernel_17348\159985506.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0901 13:11:29.049000 17348 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of t

[torch.onnx] Obtain model graph for `ONNXQueryClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ONNXQueryClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


d:\razorpay_revenue_recovery\.venv\Lib\site-packages\torch\onnx\_internal\exporter\_onnx_program.py:486: UserWarning: # The axis name: sequence_length will not be used, since it shares the same shape constraints with another axis: sequence_length.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


ONNX model exported to:
d:\razorpay_revenue_recovery\ml\models\query_classifier\onnx\model.onnx


In [130]:
import onnx 

onnx_model_check = onnx.load(str(ONNX_PATH))

onnx.checker.check_model(onnx_model_check)

print("✓ ONNX model structure is valid")

✓ ONNX model structure is valid


In [132]:
import onnxruntime as ort

session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

print("Inputs:")

for input_info in session.get_inputs():
    print(input_info.name,
          input_info.shape,
          input_info.type)

print("Outputs:")

for output_info in session.get_outputs():
    print(output_info.name,
          output_info.shape,
          output_info.type)

Inputs:
input_ids ['batch_size', 'sequence_length'] tensor(int64)
attention_mask ['batch_size', 'sequence_length'] tensor(int64)
Outputs:
intent_logits ['batch_size', 11] tensor(float)
retrieval_logits ['batch_size', 3] tensor(float)


In [133]:
sample = test_dataset[0]

sample_input_ids = sample["input_ids"].unsqueeze(0)

sample_attention_mask = sample["attention_mask"].unsqueeze(0)

with torch.no_grad():
    torch_intent, torch_retrieval = model(input_ids=sample_input_ids, attention_mask=sample_attention_mask,)

onnx_outputs = session.run(None, {"input_ids": sample_input_ids.numpy(),"attention_mask": sample_attention_mask.numpy()})

onnx_intent = onnx_outputs[0]
onnx_retrieval = onnx_outputs[1]

In [138]:
import numpy as np


intent_difference = np.max(np.abs(torch_intent.numpy() - onnx_intent))

retrieval_difference = np.max(np.abs(torch_retrieval.numpy() - onnx_retrieval))


print(
    "Max intent-logit difference:",
    intent_difference,
)

print(
    "Max retrieval-logit difference:",
    retrieval_difference,
)


Max intent-logit difference: 2.8014183e-06
Max retrieval-logit difference: 1.9073486e-06


In [140]:
torch_intent_prediction = torch.argmax(
    torch_intent,
    dim=1,
).item()

onnx_intent_prediction = int(
    np.argmax(onnx_intent, axis=1)[0]
)

torch_retrieval_prediction = torch.argmax(
    torch_retrieval,
    dim=1,
).item()

onnx_retrieval_prediction = int(
    np.argmax(onnx_retrieval, axis=1)[0]
)

print(
    "Intent:",
    torch_intent_prediction,
    "vs",
    onnx_intent_prediction,
)

print(
    "Retrieval:",
    torch_retrieval_prediction,
    "vs",
    onnx_retrieval_prediction,
)

Intent: 7 vs 7
Retrieval: 0 vs 0


In [141]:
from sklearn.metrics import accuracy_score, f1_score


all_intent_targets = []
all_intent_predictions = []

all_retrieval_targets = []
all_retrieval_predictions = []


for batch in test_loader:

    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    # ONNX Runtime inference
    onnx_outputs = session.run(
        None,
        {
            "input_ids": input_ids.numpy(),
            "attention_mask": attention_mask.numpy(),
        },
    )

    intent_logits = onnx_outputs[0]
    retrieval_logits = onnx_outputs[1]

    # Predictions
    intent_predictions = np.argmax(
        intent_logits,
        axis=1,
    )

    retrieval_predictions = np.argmax(
        retrieval_logits,
        axis=1,
    )

    # Collect targets
    all_intent_targets.extend(
        batch["intent_id"].numpy()
    )

    all_intent_predictions.extend(
        intent_predictions
    )

    all_retrieval_targets.extend(
        batch["retrieval_mode_id"].numpy()
    )

    all_retrieval_predictions.extend(
        retrieval_predictions
    )


# Calculate metrics
onnx_intent_accuracy = accuracy_score(
    all_intent_targets,
    all_intent_predictions,
)

onnx_intent_f1 = f1_score(
    all_intent_targets,
    all_intent_predictions,
    average="macro",
)

onnx_retrieval_accuracy = accuracy_score(
    all_retrieval_targets,
    all_retrieval_predictions,
)

onnx_retrieval_f1 = f1_score(
    all_retrieval_targets,
    all_retrieval_predictions,
    average="macro",
)


print("=" * 60)
print("ONNX TEST RESULTS")
print("=" * 60)

print(
    f"Intent accuracy:       "
    f"{onnx_intent_accuracy:.4f}"
)

print(
    f"Intent macro-F1:       "
    f"{onnx_intent_f1:.4f}"
)

print(
    f"Retrieval accuracy:    "
    f"{onnx_retrieval_accuracy:.4f}"
)

print(
    f"Retrieval macro-F1:    "
    f"{onnx_retrieval_f1:.4f}"
)

ONNX TEST RESULTS
Intent accuracy:       0.8476
Intent macro-F1:       0.8468
Retrieval accuracy:    0.7048
Retrieval macro-F1:    0.7057


In [143]:
TOKENIZER_DIR = ONNX_DIR / "tokenizer"

TOKENIZER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

tokenizer.save_pretrained(
    TOKENIZER_DIR
)

print(
    f"Tokenizer saved to:\n{TOKENIZER_DIR}"
)

Tokenizer saved to:
d:\razorpay_revenue_recovery\ml\models\query_classifier\onnx\tokenizer


In [144]:
import json

labels = {
    "intent": {
        str(i): label
        for i, label in enumerate(
            intent_encoder.classes_
        )
    },
    "retrieval_mode": {
        str(i): label
        for i, label in enumerate(
            retrieval_encoder.classes_
        )
    },
}

LABELS_PATH = ONNX_DIR / "labels.json"

with LABELS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        labels,
        file,
        indent=2,
    )

print(
    f"Labels saved to:\n{LABELS_PATH}"
)

Labels saved to:
d:\razorpay_revenue_recovery\ml\models\query_classifier\onnx\labels.json
